In [ ]:
# Imports ONE for dataset access and SpikeInterface for reading, preprocessing, and analyzer creation

import os
from one.api import ONE
import spikeinterface.preprocessing as spre
from spikeinterface.extractors import read_cbin_ibl
from spikeinterface.extractors import  IblSortingExtractor
from spikeinterface import create_sorting_analyzer, load_sorting_analyzer

In [ ]:
# Absolute path to raw CBIN probe data — change to your local mount.
data_path =  r'E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\raw_ephys_data\probe00'

In [ ]:
# Load the CBIN recording using SpikeInterface RecordingExtractor.
recording = read_cbin_ibl(folder_path=data_path)
print(recording)

CompressedBinaryIblExtractor: 384 channels - 30000.294444 Hz - 1 segments - 164,856,082 samples 
                              5,495.15s (1.53 hours) - int16 dtype - 117.91 GiB


In [ ]:
# Apply preprocessing: highpass filter at 300 Hz, correct phase shifts, then common reference across channels.
recording = spre.highpass_filter(recording, 300)
recording = spre.phase_shift(recording)
recording = spre.common_reference(recording)

In [ ]:
one = ONE(base_url='https://alyx.internationalbrainlab.org', username = '', password = '', silent=True)
print(one)

One (online, https://alyx.internationalbrainlab.org)


In [ ]:
pid = '89a91bda-cce2-4a56-b226-0d56c8f50577'

One (online, https://alyx.internationalbrainlab.org)


In [ ]:
# Create an IBL sorting extractor that queries Alyx/ONE for the given pid. This returns spike times and unit metadata
sorting = IblSortingExtractor(one=one, pid=pid)
print(sorting)

local file size mismatch on dataset: wittenlab/Subjects/dop_47/2022-06-05/001/alf/probe00/pykilosort/clusters.metrics.pqt
(S3) E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\alf\probe00\pykilosort\clusters.metrics.pqt: 100%|██████████| 28.0k/28.0k [00:00<00:00, 82.1kB/s]
(S3) E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\alf\probe00\pykilosort\_av_clusters.channels.npy: 100%|██████████| 1.13k/1.13k [00:00<00:00, 4.74kB/s]
(S3) E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\alf\probe00\pykilosort\_av_clusters.depths.npy: 100%|██████████| 1.13k/1.13k [00:00<00:00, 4.34kB/s]
(S3) E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\alf\probe00\pykilosort\channels.mlapdv.npy: 100%|██████████| 9.34k/9.34k [00:00<00:00, 39.6kB/s]
(S3) E:\ONE\alyx.internationalbrainlab.org\wittenlab\Subjects\dop_47\2022-06-05\001\alf\probe00\pykilosort\channels.brainLocationIds_ccf_2017.n

IblSortingExtractor: 125 units - 1 segments - 30.0kHz


In [ ]:
# Create a SortingAnalyzer: computes and stores templates, waveforms, metrics, and other intermediate binary files to analyzer_folder. 
analyzer_folder = os.path.join(data_path, 'sorting_analyzer_folder')
sorting_analyzer = create_sorting_analyzer(sorting=sorting,
                                    recording=recording,
                                    format="binary_folder",
                                    folder=analyzer_folder,
                                    overwrite=True)
print(sorting_analyzer)

In [ ]:
# Reload an existing SortingAnalyzer from disk; useful to avoid recomputing expensive steps
analyzer_folder = os.path.join(data_path, 'sorting_analyzer_folder')
sorting_analyzer = load_sorting_analyzer(folder=analyzer_folder)
print(sorting_analyzer)

SortingAnalyzer: 384 channels - 125 units - 1 segments - binary_folder - sparse - has recording
Loaded 12 extensions: correlograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, spike_locations, templates, template_metrics, template_similarity, unit_locations, waveforms


In [ ]:
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations',
                            'spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])
